# BTLON02 — Notebook 02: Thực nghiệm Nhóm 1 (Baseline CNN, Data Augmentation, Activation & Class Weight)
- **Phụ trách**: Thành viên 1 — Lý Hồng Hưng (`3123411132`)
- **Môi trường khuyến nghị**: Kaggle Notebook — bật **GPU T4**, **Internet: On**, Add Input: **FER2013** (`deadskull7/fer2013` hoặc `msambare/fer2013`).
- **6 thực nghiệm có kiểm soát (100% PyTorch)**:
  1. `Exp1_1`: `SmallCNN` — Không Augmentation, `ReLU`, CrossEntropy *(Baseline 0)*
  2. `Exp1_2`: `SmallCNN` — Có Augmentation, `ReLU`, CrossEntropy *(Tác động Augmentation)*
  3. `Exp1_3`: `MediumCNN` — Có Augmentation, `ReLU`, CrossEntropy *(So sánh độ sâu)*
  4. `Exp1_4`: `MediumCNN` — Có Augmentation, `LeakyReLU`, CrossEntropy *(So sánh Activation)*
  5. `Exp1_5`: `MediumCNN` — Có Augmentation, `Sigmoid`, CrossEntropy *(Minh chứng Vanishing Gradient)*
  6. `Exp1_6`: `MediumCNN` — Có Augmentation, `ReLU`, **Weighted CrossEntropy** *(Xử lý mất cân bằng lớp `Disgust`)*

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HongHungLy/SGU26K2_HocSau_S5_DCT123C4_Nhom666HocSau.git"
BRANCH = "develop"
PROJECT_SUBDIR = "BTLON02_Emotion_Ly Hong Hung_3123411132"

if Path("/kaggle/working").exists():
    repo_dir = Path("/kaggle/working/repo")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "origin", BRANCH], check=True)
    PROJECT_DIR = repo_dir / PROJECT_SUBDIR
else:
    # Chạy local từ thư mục notebooks/ hoặc thư mục gốc dự án
    cwd = Path.cwd()
    PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Working directory:", Path.cwd())

## 1. Kiểm tra GPU & Môi trường PyTorch

In [ ]:
import torch
from src.utils import get_device
from src.dataset import discover_fer2013_source

print("PyTorch version:", torch.__version__)
device = get_device()
print("FER2013 source:", discover_fer2013_source())

## 2. Chạy toàn bộ Thực nghiệm Nhóm 1 (`exp1_baseline.py`)
*(Trên GPU T4, toàn bộ 6 thực nghiệm × 45 epochs mất khoảng ~25–35 phút. Muốn test nhanh trước có thể đổi `EPOCHS = 2` hoặc truyền `--fast_test`.)*

In [ ]:
EPOCHS = 45
BATCH_SIZE = 64
LR = 1e-3
PATIENCE = 10

!python exp1_baseline.py --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --patience {PATIENCE} --seed 42

## 3. Xem bảng tổng hợp kết quả & Biểu đồ so sánh Thực nghiệm Nhóm 1

In [ ]:
import pandas as pd
from IPython.display import Image as IPyImage, display

df_sum = pd.read_csv("logs/exp1_summary.csv")
display(df_sum[[
    "exp_name", "total_params", "best_epoch", "best_val_acc",
    "best_val_loss", "test_acc", "test_macro_f1", "test_disgust_recall", "train_time_sec"
]])

comp_fig = Path("figures/exp1_baseline_comparison.png")
if comp_fig.exists():
    display(IPyImage(filename=str(comp_fig)))

## 4. Hiển thị đường cong Accuracy & Loss của từng thực nghiệm

In [ ]:
for img_path in sorted(Path("figures").glob("Exp1_*_curves.png")):
    print(">>>", img_path.name)
    display(IPyImage(filename=str(img_path)))

## 5. Đóng gói kết quả để tải về
- **`tv1_baseline_outputs.zip`**: Chứa `logs/` và `figures/` (nhẹ -> tải về giải nén vào repo và **push lên nhánh `develop`**).
- **`tv1_checkpoints.zip`**: Chứa các file trọng số `.pth` (nặng -> tải về up lên Google Drive chia sẻ cho TV3 test `evaluate.py` & `visualize.py`, **không push lên Git**).

In [ ]:
import zipfile

out_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

out_zip = out_root / "tv1_baseline_outputs.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["figures", "logs"]:
        for f in Path(folder).rglob("*"):
            if f.is_file() and f.name != ".gitkeep":
                zf.write(f, arcname=str(f))
print("Đã tạo:", out_zip)

ckpt_zip = out_root / "tv1_checkpoints.zip"
with zipfile.ZipFile(ckpt_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in Path("checkpoints").glob("*.pth"):
        zf.write(f, arcname=str(f))
print("Đã tạo:", ckpt_zip)